# 06 · Evaluation harness (S1-S4 and the ablation)

**Purpose.** Given only the papers a real related-work section cites, regenerate that section and score it: ROUGE and BERTScore against the human text, plus citation coverage, citation correctness and NLI faithfulness, with bootstrap 95% CIs. Same model and settings across every system.

| | |
|---|---|
| **Inputs** | `data/rw_pairs.jsonl` test split (frozen), `data/multixscience_test.jsonl`, `models/scibert_relations/`, optionally `models/distilled_extractor/` |
| **Outputs** | `data/eval_generations_*.jsonl`, `results/eval_arxiv.json`, `results/eval_multixscience.json`, `figures/*.png` |
| **Expected runtime (Colab T4)** | 2-5 h for S1-S4 over 100 examples (API-bound); scoring ~20 min |
| **Required secrets** | `GROQ_API_KEY` |
| **GPU** | T4 recommended (NLI, BERTScore, SciBERT) |

Everything is cached under `/content/drive/MyDrive/litreview/` and every stage is
**resumable**: if Colab disconnects, re-run all cells and it continues where it
stopped. Only the cell marked CONFIG needs editing.


In [ ]:
# --- Colab setup: Drive, project code, secrets. Safe to re-run. ---
import json, os, subprocess, sys, zipfile
from pathlib import Path
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = "/content/drive/MyDrive/litreview" if ON_COLAB else os.path.abspath("..")
os.makedirs(ROOT, exist_ok=True)
os.environ["LITREVIEW_ROOT"] = ROOT
REPO = os.path.join(ROOT, "repo")

# Optional: a git URL to clone the project from. Leave blank to use the folder or
# the zip on Drive instead.
REPO_URL = ""

def _find_project() -> str | None:
    """Locate the project code: an existing folder, a zip on Drive, or a clone."""
    if os.path.exists(os.path.join(REPO, "pyproject.toml")):
        return REPO
    # a litreview_*.zip uploaded anywhere under MyDrive/litreview
    zips = sorted(Path(ROOT).glob("litreview*.zip")) + sorted(Path(ROOT).glob("**/litreview*.zip"))
    for z in zips:
        print(f"extracting {z.name} ...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(ROOT)
        # the zip may contain a top-level folder; find the pyproject inside
        for cand in [REPO, ROOT] + [str(p.parent) for p in Path(ROOT).glob("*/pyproject.toml")]:
            if os.path.exists(os.path.join(cand, "pyproject.toml")):
                return cand
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=False)
        if os.path.exists(os.path.join(REPO, "pyproject.toml")):
            return REPO
    return None

if ON_COLAB:
    found = _find_project()
    if not found:
        raise SystemExit(
            "Could not find the project code. Do one of these, then re-run this cell:
"
            "  1. upload litreview_colab.zip into MyDrive/litreview/ (easiest), or
"
            "  2. copy the project folder to MyDrive/litreview/repo, or
"
            "  3. set REPO_URL above to a git URL.
"
            "The notebooks deliberately contain no second copy of the code.")
    REPO = found
    print("project code:", REPO)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", f"{REPO}[ml,eval]"],
                   check=True)

import litreview
from litreview.config import SETTINGS, paths, secret
P = paths().mkdirs()
print("data root:", P.root)
print("already present:", {f.name: f.stat().st_size // 1024 for f in sorted(P.data.glob("*.jsonl"))})


In [ ]:
# ======================= CONFIG (the only cell to edit) =======================
PROVIDER  = "groq"
MODEL     = ""
SYSTEMS   = ["S1", "S2", "S3", "S3_no_relations"]   # add "S4" once notebook 05 has run
LIMIT     = 100              # test examples; the whole split if smaller
BENCHMARK = "arxiv"          # "arxiv" | "multixscience"
DISTILLED = ""               # e.g. str(P.models / "distilled_extractor") to enable S4
BERTSCORE = True
# ==============================================================================


In [ ]:
# Provider comes from one config. Keys are read from Colab secrets, never hard-coded.
import os
os.environ["LITREVIEW_PROVIDER"] = PROVIDER
if MODEL:
    os.environ["LITREVIEW_MODEL"] = MODEL

from litreview.config import Settings
from litreview.llm import install_call_logger, probe, usage_summary
import litreview.config as _cfg
_cfg.SETTINGS = Settings()                      # re-read after setting the env vars
import litreview.llm as _llm
_llm.SETTINGS = _cfg.SETTINGS

install_call_logger(P.results / "llm_calls.jsonl")
print(json.dumps(_cfg.SETTINGS.describe(), indent=1))
print(json.dumps(probe(_cfg.SETTINGS), indent=1))


### The benchmark is frozen

The test split comes from a submission window that is later than the training
window **and** later than the knowledge cutoff of every model being evaluated
(`docs/leakage.md`). Nothing in this project is tuned on it: the Verifier's
thresholds, the classifier's dev split and the gold label set all come from the
train split.

In [ ]:
from litreview.eval.runner import load_benchmark
bench_path = (P.data / "rw_pairs.jsonl" if BENCHMARK == "arxiv"
              else P.data / f"multixscience_test.jsonl")
bench = load_benchmark(bench_path, split="test", limit=LIMIT)
print(f"{len(bench)} examples from {bench_path.name}")
import pandas as pd
print(pd.DataFrame([{"arxiv_id": b["arxiv_id"], "n_cited": len(b["cited"]),
                     "rw_words": len(b["rw_text"].split()),
                     "published": b.get("published")} for b in bench]).describe().round(1))


### The systems

| | |
|---|---|
| **S1** | one well-prompted LLM call over the same cited abstracts |
| **S2** | crew without the Verifier: extract, then write |
| **S3** | full crew: extract → verify → relations → map → gaps → write → verify/revise |
| **S4** | S3 with the distilled extractor from notebook 05 |
| **S3_no_relations** | ablation: S3 with the relation classifier switched off |

Generation is cached per (system, example), so a run stopped by a rate limit
resumes. Scoring is a separate pass over that cache, so metrics can be recomputed
without regenerating any text.

In [ ]:
from tqdm.auto import tqdm
from litreview.eval.runner import cost_table, generate, save, score, table
from litreview.relations.scibert import RelationClassifier
from litreview.verify import Verifier

verifier = Verifier()
classifier = (RelationClassifier(P.models / "scibert_relations")
              if (P.models / "scibert_relations").exists() else None)
distilled = None
if DISTILLED:
    from litreview.extract import DistilledExtractor
    distilled = DistilledExtractor(DISTILLED)

cache = P.data / f"eval_generations_{BENCHMARK}.jsonl"
gen = generate(bench, SYSTEMS, cache, verifier=verifier, classifier=classifier,
               distilled=distilled, progress=lambda js: tqdm(js, desc="generate"))
print(json.dumps(gen, indent=1))


In [ ]:
rep = score(bench, cache, nli=verifier.nli, with_bertscore=BERTSCORE,
            progress=lambda rs: tqdm(rs, desc="score"))
rep["benchmark"] = {"path": str(bench_path), "split": "test",
                    "n_examples": len(bench), "source": BENCHMARK}
rep["settings"] = _cfg.SETTINGS.describe()
rep["main_table"] = table(rep["aggregates"])
rep["cost_table"] = cost_table(rep["aggregates"], usage_summary())
save(rep, P.results / f"eval_{BENCHMARK}.json")

print("== main table ==")
display(pd.DataFrame(rep["main_table"]))
print("== cost / latency ==")
display(pd.DataFrame(rep["cost_table"]))


### Qualitative side by side

Two or three examples, human section against each system's output. These are what
the error analysis in the report is built from.

In [ ]:
from IPython.display import Markdown, display
rows = [json.loads(l) for l in cache.open()]
by_id = {}
for r in rows:
    by_id.setdefault(r["arxiv_id"], {})[r["system"]] = r
for aid in list(by_id)[:3]:
    gold = next(b for b in bench if b["arxiv_id"] == aid)
    out = [f"### {aid} — {(gold.get('title') or '')[:100]}",
           f"**Human section ({len(gold['rw_text'].split())} words)**\n\n{gold['rw_text'][:1200]}"]
    for s in SYSTEMS:
        r = by_id[aid].get(s) or {}
        txt = (r.get("review") or {}).get("markdown") or f"_({r.get('error') or 'not run'})_"
        out.append(f"**{s}**\n\n{txt[:1200]}")
    display(Markdown("\n\n---\n\n".join(out)))


### Figures

Every figure is generated from `results/*.json` by one script, so no figure can
show a number that is not in a results file.

In [ ]:
import subprocess, sys
SCRIPTS = (Path(REPO) / "scripts") if ON_COLAB else Path("../scripts")
subprocess.run([sys.executable, str(SCRIPTS / "usage_report.py")], check=False)
subprocess.run([sys.executable, str(SCRIPTS / "make_figures.py")], check=False)
print(json.dumps(json.loads((P.figures / "figures_manifest.json").read_text()), indent=1)[:2000])


### Done

`results/` now holds every number the report quotes and `figures/` every figure it
embeds. Build the report with `python scripts/build_report.py`.